In [125]:
import os
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# 1. Data load
df = pd.read_csv(r"D:\Human voice multi classification\audio_dataset.csv")
x = df.drop("label", axis=1).values
y_raw = df["label"].values

# 2. Label encode
le = LabelEncoder()
y = le.fit_transform(y_raw)

# 3. Train/test split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Scale (fit sirf train par)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# 5. Model train
model = XGBClassifier(n_estimators=200, random_state=42)
model.fit(x_train, y_train)

# 6. Evaluate
pred = model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=le.classes_.astype(str)))

# 7. Auto-save
SAVE_DIR = r"D:\Human voice multi classification\saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)
joblib.dump(model, os.path.join(SAVE_DIR, "xgb_model.pkl"))
joblib.dump(scaler, os.path.join(SAVE_DIR, "scaler.pkl"))
joblib.dump(le, os.path.join(SAVE_DIR, "label_encoder.pkl"))
print("Saved in:", SAVE_DIR)

Accuracy: 0.95
              precision    recall  f1-score   support

           0       0.94      0.75      0.83        67
           1       0.95      0.99      0.97       333

    accuracy                           0.95       400
   macro avg       0.95      0.87      0.90       400
weighted avg       0.95      0.95      0.95       400

Saved in: D:\Human voice multi classification\saved_models


In [126]:
print('model' in globals(), 'scaler' in globals(), 'le' in globals())

True True True


In [127]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score,ConfusionMatrixDisplay,classification_report
import matplotlib.pyplot as plt

In [128]:
import pandas as pd
data = pd.read_csv(r'D:\Human voice multi classification\dataset\train.csv')

In [129]:
data

,file_id,gender
0,common_voice_hi_26204093,male
1,common_voice_hi_26043935,male
2,common_voice_hi_24632150,male
3,common_voice_hi_24359471,female
4,common_voice_hi_24639127,male
...,...,...
1993,common_voice_hi_24225356,male
1994,common_voice_hi_26114822,male
1995,common_voice_hi_25998821,male
1996,common_voice_hi_23849314,male


#Data is highly Imbalance

In [130]:
gender_counts = data.groupby('gender').size().reset_index(name='count')
display(gender_counts)

,gender,count
0,female,334
1,male,1664


#picking a Mini data-set

In [131]:
female_data = data[data['gender'] == 'female'].sample(n=334, random_state=42)
male_data = data[data['gender'] == 'male'].sample(n=650, random_state=42)

new_data = pd.concat([female_data, male_data])

display(new_data)

,file_id,gender
186,common_voice_hi_25154337,female
1800,common_voice_hi_24360599,female
412,common_voice_hi_25154389,female
1116,common_voice_hi_26239091,female
323,common_voice_hi_24969916,female
...,...,...
103,common_voice_hi_25982521,male
48,common_voice_hi_26354239,male
1933,common_voice_hi_26204010,male
152,common_voice_hi_23852747,male


In [132]:
new_data

,file_id,gender
186,common_voice_hi_25154337,female
1800,common_voice_hi_24360599,female
412,common_voice_hi_25154389,female
1116,common_voice_hi_26239091,female
323,common_voice_hi_24969916,female
...,...,...
103,common_voice_hi_25982521,male
48,common_voice_hi_26354239,male
1933,common_voice_hi_26204010,male
152,common_voice_hi_23852747,male


#Extract Features from the Audio Clips

In [133]:
import os
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

# === Step 1: Path setup ===
AUDIO_PATH = r"D:\Human voice multi classification\dataset\train"
SAMPLE_RATE = 22050
N_MFCC = 13

# === Step 2: Use the labels already loaded in the notebook ===
df_labels = data.copy()
print("✅ Labels loaded:", df_labels.shape)
print(df_labels.head())

# === Step 3: Encode gender labels ===
le = LabelEncoder()
df_labels["gender_encoded"] = le.fit_transform(df_labels["gender"])
print("\n🔹 Label encoding mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# === Step 4: Extract MFCC function ===
def extract_mfcc(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    return mfcc

# === Step 5: Find available audio files ===
audio_extensions = (".mp3", ".wav", ".flac", ".ogg")
audio_files = {}

if os.path.isdir(AUDIO_PATH):
    for filename in os.listdir(AUDIO_PATH):
        file_id, extension = os.path.splitext(filename)
        if extension.lower() in audio_extensions:
            audio_files[file_id] = os.path.join(AUDIO_PATH, filename)

print(f"🔹 Audio files found: {len(audio_files)}")

# === Step 6: Find max frame length ===
max_len = 0
for file_id in tqdm(df_labels["file_id"], desc="Finding max frame length"):
    file_path = audio_files.get(file_id)

    if file_path:
        mfcc = extract_mfcc(file_path)
        max_len = max(max_len, mfcc.shape[1])

print(f"🔹 Maximum frame length found: {max_len}")

# === Step 7: Pad/Trim function ===
def pad_mfcc(mfcc, max_len):
    if mfcc.shape[1] < max_len:
        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode="constant")
    else:
        mfcc = mfcc[:, :max_len]
    return mfcc

# === Step 8: Process all audio files ===
data_rows = []
missing_files = []

for idx, row in tqdm(
    df_labels.iterrows(),
    total=len(df_labels),
    desc="Processing audio"
):
    file_id = row["file_id"]
    file_path = audio_files.get(file_id)

    if file_path is None:
        missing_files.append(file_id)
        continue

    mfcc = extract_mfcc(file_path)
    mfcc = pad_mfcc(mfcc, max_len)
    features = mfcc.flatten()

    # Combine features + label
    data_rows.append(np.append(features, row["gender_encoded"]))

# === Step 9: Convert to DataFrame ===
if data_rows:
    feature_count = len(data_rows[0]) - 1
    df_features = pd.DataFrame(
        data_rows,
        columns=[f"f{i}" for i in range(feature_count)] + ["label"]
    )
else:
    df_features = pd.DataFrame(columns=["label"])
    print("⚠️ No matching audio files were found.")
    print(f"Check that AUDIO_PATH is correct: {AUDIO_PATH}")

print("\n✅ Final dataset shape:", df_features.shape)

# === Step 10: Save to CSV ===
OUTPUT_CSV = "audio_features_gender_dataset_test.csv"
df_features.to_csv(OUTPUT_CSV, index=False)
print(f"💾 Dataset saved successfully as: {OUTPUT_CSV}")

if missing_files:
    print("\n⚠️ Missing files:", missing_files[:10], "...")


✅ Labels loaded: (1998, 2)
                    file_id  gender
0  common_voice_hi_26204093    male
1  common_voice_hi_26043935    male
2  common_voice_hi_24632150    male
3  common_voice_hi_24359471  female
4  common_voice_hi_24639127    male

🔹 Label encoding mapping: {'female': np.int64(0), 'male': np.int64(1)}
🔹 Audio files found: 0


Finding max frame length: 100%|██████████| 1998/1998 [00:00<00:00, 334927.44it/s]

🔹 Maximum frame length found: 0



Processing audio: 100%|██████████| 1998/1998 [00:00<00:00, 15250.60it/s]

⚠️ No matching audio files were found.
Check that AUDIO_PATH is correct: D:\Human voice multi classification\dataset\train

✅ Final dataset shape: (0, 1)
💾 Dataset saved successfully as: audio_features_gender_dataset_test.csv

⚠️ Missing files: ['common_voice_hi_26204093', 'common_voice_hi_26043935', 'common_voice_hi_24632150', 'common_voice_hi_24359471', 'common_voice_hi_24639127', 'common_voice_hi_25983363', 'common_voice_hi_23832738', 'common_voice_hi_25154335', 'common_voice_hi_26080943', 'common_voice_hi_24360528'] ...


#Recorrection the Target Label

In [134]:
import zipfile

# OUTPUT_CSV was saved in the notebook's current directory
csv_path = os.path.abspath(OUTPUT_CSV)

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Feature file not found: {csv_path}")

df = pd.read_csv("audio_dataset.csv")

if df.empty or len(df.columns) <= 1:
    audio_extensions = (".mp3", ".wav", ".flac", ".ogg", ".opus", ".m4a")
    configured_path = os.path.abspath(AUDIO_PATH)
    audio_files = {}

    for root, _, files in os.walk(configured_path):
        for filename in files:
            stem, extension = os.path.splitext(filename)

            if extension.lower() in audio_extensions:
                audio_files[stem.lower()] = os.path.join(root, filename)

    print(f"Audio files found: {len(audio_files)}")

    matched_labels = []

    for _, row in df_labels.iterrows():
        file_path = audio_files.get(str(row["file_id"]).strip().lower())

        if file_path:
            matched_labels.append((row, file_path))

    if not matched_labels:
        dataset_root = os.path.dirname(configured_path)
        zip_candidates = [
            os.path.join(dataset_root, name)
            for name in os.listdir(dataset_root)
            if name.lower().endswith(".zip")
            and name.lower().startswith("train")
        ]

        if zip_candidates:
            print(f"Extracting audio archive: {zip_candidates[0]}")

            os.makedirs(configured_path, exist_ok=True)

            with zipfile.ZipFile(zip_candidates[0], "r") as archive:
                archive.extractall(configured_path)

            audio_files = {}

            for root, _, files in os.walk(configured_path):
                for filename in files:
                    stem, extension = os.path.splitext(filename)

                    if extension.lower() in audio_extensions:
                        audio_files[stem.lower()] = os.path.join(root, filename)

            matched_labels = []

            for _, row in df_labels.iterrows():
                file_path = audio_files.get(
                    str(row["file_id"]).strip().lower()
                )

                if file_path:
                    matched_labels.append((row, file_path))

            print(f"Audio files found after extraction: {len(audio_files)}")
            print(f"Matched CSV records: {len(matched_labels)}")

    if not matched_labels:
        raise ValueError(
            "No matching audio files were found. Extract the train ZIP archive "
            "into this folder:\n"
            f"{configured_path}\n"
            "The audio filenames must match the CSV file_id values."
        )

    max_len = 0

    for _, file_path in matched_labels:
        mfcc = extract_mfcc(file_path)
        max_len = max(max_len, mfcc.shape[1])

    data_rows = []

    for row, file_path in tqdm(matched_labels, desc="Extracting features"):
        mfcc = pad_mfcc(extract_mfcc(file_path), max_len)
        data_rows.append(
            np.append(mfcc.flatten(), row["gender_encoded"])
        )

    feature_count = len(data_rows[0]) - 1

    df_features = pd.DataFrame(
        data_rows,
        columns=[f"f{i}" for i in range(feature_count)] + ["label"]
    )

    df_features.to_csv(csv_path, index=False)
    df = df_features

    print(f"Regenerated feature dataset: {df.shape}")

print(f"Loaded dataset: {df.shape}")
display(df.head())
df

Loaded dataset: (1998, 5708)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f5698,f5699,f5700,f5701,f5702,f5703,f5704,f5705,f5706,label
0,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-455.466614,-314.775024,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-518.867676,-340.706879,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,-584.527344,-530.542542,-476.353546,-465.709595,-460.492523,-456.223358,-450.025024,-445.821930,-443.131165,-451.579071,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,-529.723267,-469.574524,-395.072388,-384.778290,-408.282806,-415.518890,-419.215057,-429.356323,-452.158539,-457.296387,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f5698,f5699,f5700,f5701,f5702,f5703,f5704,f5705,f5706,label
0,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-455.466614,-314.775024,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-518.867676,-340.706879,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,-584.527344,-530.542542,-476.353546,-465.709595,-460.492523,-456.223358,-450.025024,-445.821930,-443.131165,-451.579071,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,-529.723267,-469.574524,-395.072388,-384.778290,-408.282806,-415.518890,-419.215057,-429.356323,-452.158539,-457.296387,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1993,-674.801453,-674.801453,-653.942444,-642.859985,-658.755249,-661.210266,-664.422119,-666.317017,-664.966248,-663.801514,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1994,-568.691711,-568.691711,-568.691711,-568.691711,-568.691711,-568.691711,-568.691711,-568.691711,-497.911133,-346.384064,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1995,-765.633057,-709.184814,-653.840088,-643.214966,-654.505493,-660.623291,-666.057007,-680.841980,-693.213806,-710.703186,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1996,-565.381165,-535.522644,-511.489624,-512.039978,-511.222168,-507.454773,-504.868011,-504.518616,-511.605469,-513.085999,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [135]:
import os

print("Audio folder ki files:")
# Locate the directory that actually contains the audio files
audio_dir = None

for root, _, files in os.walk(base):
    if any(file.lower().endswith((".mp3", ".wav", ".flac", ".ogg"))
           for file in files):
        audio_dir = root
        break

if audio_dir is None:
    raise FileNotFoundError(f"No audio files found under: {base}")

# Update the path used by the feature-extraction cells
AUDIO_PATH = audio_dir

print("Audio folder:", AUDIO_PATH)
print(os.listdir(AUDIO_PATH)[:5])

print("CSV ke file_id:")
print(df_labels["file_id"].head().tolist())

Audio folder ki files:
Audio folder: D:\Human voice multi classification\dataset\train\train
['common_voice_hi_23795240.mp3', 'common_voice_hi_23795241.mp3', 'common_voice_hi_23795244.mp3', 'common_voice_hi_23795249.mp3', 'common_voice_hi_23795250.mp3']
CSV ke file_id:
['common_voice_hi_26204093', 'common_voice_hi_26043935', 'common_voice_hi_24632150', 'common_voice_hi_24359471', 'common_voice_hi_24639127']


In [136]:
import os

base = r"D:\Human voice multi classification\dataset"
print("Dataset folder ke andar:", os.listdir(base))


Dataset folder ke andar: ['test-20261023T104431Z-1-001.zip', 'test.csv', 'train', 'train-20261023T104431Z-1-001.zip', 'train.csv']


In [137]:
import os

base = r"D:\Human voice multi classification\dataset"
print(os.listdir(base))

['test-20261023T104431Z-1-001.zip', 'test.csv', 'train', 'train-20261023T104431Z-1-001.zip', 'train.csv']


In [138]:
# Check the current data type of the 'label' column
print("Original data type of 'label' column:", df['label'].dtype)

# Change the data type of the 'label' column to integer
df['label'] = df['label'].astype(int)

# Check the new data type of the 'label' column
print("New data type of 'label' column:", df['label'].dtype)

# Display the updated DataFrame (optional)
display(df.head())

Original data type of 'label' column: int64
New data type of 'label' column: int64


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,...,f5698,f5699,f5700,f5701,f5702,f5703,f5704,f5705,f5706,label
0,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-574.782532,-455.466614,-314.775024,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-586.120544,-518.867676,-340.706879,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,-574.140564,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,-584.527344,-530.542542,-476.353546,-465.709595,-460.492523,-456.223358,-450.025024,-445.821930,-443.131165,-451.579071,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,-529.723267,-469.574524,-395.072388,-384.778290,-408.282806,-415.518890,-419.215057,-429.356323,-452.158539,-457.296387,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


#save it again

In [139]:
df.to_csv("audio_dataset.csv", index=False)

Machine Learning Part

In [140]:
from sklearn.linear_model import LinearRegression, LogisticRegressionCV,RidgeClassifierCV,LassoCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC,LinearSVC
from sklearn.neighbors import KNeighborsClassifier
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score,ConfusionMatrixDisplay,classification_report
import matplotlib.pyplot as plt

In [141]:
df = pd.read_csv(r"D:\Human voice multi classification\audio_dataset.csv")
x = df.drop("label", axis=1).values
y = df["label"].values


In [142]:
le= LabelEncoder()
y = le.fit_transform(y)

st = StandardScaler()
x = st.fit_transform(x)

In [143]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [144]:
m1 = LogisticRegressionCV()
m2 = RidgeClassifierCV()
m3 = DecisionTreeClassifier()
m4 = SVC()
m5 = LinearSVC()
m6 = KNeighborsClassifier()

In [145]:
m1 = m1.fit(x_train,y_train)
m2 = m2.fit(x_train,y_train,)
m3 = m3.fit(x_train,y_train)
m4 = m4.fit(x_train,y_train)
m5 = m5.fit(x_train,y_train)
m6 = m6.fit(x_train,y_train)

d:\Human voice multi classification\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2071: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
d:\Human voice multi classification\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2116: FutureWarning: The default value of the parameter 'scoring' will change from None, i.e. accuracy, to 'neg_log_loss' in version 1.11. To silence this warning, explicitly set the scoring parameter: scoring='neg_log_loss' for the new, scoring='accuracy' or scoring=None for the old default.
  warnings.warn(
d:\Human voice multi classification\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:2129: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remo

In [146]:
y1 = m1.predict(x_test)
y2 = m2.predict(x_test)
y3 = m3.predict(x_test)
y4 = m4.predict(x_test)
y5 = m5.predict(x_test)
y6 = m6.predict(x_test)

In [147]:
accuracy_score1 = accuracy_score(y_test,y1) # LogisticRegressionCV
accuracy_score2 = accuracy_score(y_test,y2) # RidgeClassifierCV
accuracy_score3 = accuracy_score(y_test,y3) # DecisionTreeClassifier
accuracy_score4 = accuracy_score(y_test,y4) # SVC
accuracy_score5 = accuracy_score(y_test,y5) # LinearSVC
accuracy_score6 = accuracy_score(y_test,y6) # KNeighborsClassifier


print(f"accuracy_score1 LogisticRegressionCV: {accuracy_score1}")
print(f"accuracy_score2 RidgeClassifierCV: {accuracy_score2}")
print(f"accuracy_score3 DecisionTreeClassifier: {accuracy_score3}")
print(f"accuracy_score4 SVC: {accuracy_score4}")
print(f"accuracy_score5 LinearSVC: {accuracy_score5}")
print(f"accuracy_score6 KNeighborsClassifier: {accuracy_score6}")


accuracy_score1 LogisticRegressionCV: 0.94
accuracy_score2 RidgeClassifierCV: 0.86
accuracy_score3 DecisionTreeClassifier: 0.8475
accuracy_score4 SVC: 0.9425
accuracy_score5 LinearSVC: 0.83
accuracy_score6 KNeighborsClassifier: 0.9125


In [148]:
x = m1.predict(x_test[0].reshape(1, -1))
gender = le.inverse_transform(x)
gender[0]

np.int64(1)